# Focal Loss + sqrt_rw

Focal Loss focuses on hard examples, reducing loss for well-classified samples.

In [1]:
import os, json, numpy as np, pandas as pd, torch, joblib
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
np.random.seed(42); torch.manual_seed(42)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
df_train = pd.read_csv(BASE_DIR / "data/processed/train.csv")
df_val = pd.read_csv(BASE_DIR / "data/processed/val.csv")
df_test = pd.read_csv(BASE_DIR / "data/processed/test.csv")

mapping = pd.read_csv(BASE_DIR / "data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]: df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)
print(f"Train: {len(df_train)}, Labels: {num_labels}")

Train: 16530, Labels: 9


In [3]:
city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)
print(f"Weight range: {df_train['sample_weight'].min():.3f} - {df_train['sample_weight'].max():.3f}")

Weight range: 0.148 - 1.470


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize(b): return tokenizer(b["resume_text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(df_train[["resume_text", "y", "sample_weight"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y"]]).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5510/5510 [01:00<00:00, 90.53 examples/s]


In [5]:
class FocalLossTrainer(Trainer):
    """Focal Loss: (1 - p_t)^gamma * CE, focuses on hard examples"""
    
    def __init__(self, *args, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        print(f"Focal Loss: gamma={gamma}")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Focal Loss computation
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        # Apply sample weights
        if sample_weight is not None:
            focal_loss = focal_loss * sample_weight.to(focal_loss.dtype)
        
        loss = focal_loss.mean()
        return (loss, outputs) if return_outputs else loss

In [6]:
MODEL_NAME = "bert_focal_loss"
GAMMA = 2.0

args = TrainingArguments(
    output_dir=f"./models/{MODEL_NAME}", learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=2, weight_decay=0.01, logging_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy", remove_unused_columns=False
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "macro_f1": f1_score(p.label_ids, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
trainer = FocalLossTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                           compute_metrics=compute_metrics, gamma=GAMMA)

print(f"Training: Focal Loss gamma={GAMMA} + sqrt_rw, 2 epochs")
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 201.76it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Focal Loss: gamma=2.0
Training: Focal Loss gamma=2.0 + sqrt_rw, 2 epochs


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.387456,0.925974,0.524864,0.512789
2,0.348603,0.810599,0.601815,0.626807


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.95s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.80s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

TrainOutput(global_step=4134, training_loss=0.44770622437970325, metrics={'train_runtime': 15475.7099, 'train_samples_per_second': 2.136, 'train_steps_per_second': 0.267, 'total_flos': 2174749547351040.0, 'train_loss': 0.44770622437970325, 'epoch': 2.0})

In [7]:
preds = trainer.predict(test_ds)
y_true, y_pred = preds.label_ids, np.argmax(preds.predictions, axis=1)
print(f"\nTEST: Acc={accuracy_score(y_true, y_pred):.4f}, F1={f1_score(y_true, y_pred, average='macro'):.4f}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



TEST: Acc=0.5926, F1=0.6134


In [8]:
df_eval = df_test.copy()
df_eval["y_true"], df_eval["y_pred"] = y_true, y_pred

def ovr_rates(df, grp, nc):
    groups = sorted(df[grp].dropna().unique())
    tpr, support = np.zeros((len(groups), nc)), np.zeros((len(groups), nc))
    for gi, g in enumerate(groups):
        dg = df[df[grp] == g]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(nc):
            pm = (yt == c)
            TP, FN = np.sum((yp == c) & pm), np.sum((yp != c) & pm)
            support[gi, c] = pm.sum()
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return tpr, support

def gaps(tpr, sup, ms=30):
    g = []
    for c in range(tpr.shape[1]):
        col = tpr[sup[:, c] >= ms, c]
        col = col[~np.isnan(col)]
        g.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    g = np.array(g); v = g[~np.isnan(g)]
    return v.max() if len(v) else np.nan, v.mean() if len(v) else np.nan

tpr, sup = ovr_rates(df_eval, "city_group", num_labels)
tw, tm = gaps(tpr, sup, 30)
print(f"FAIRNESS (robust): worst={tw:.4f}, macro={tm:.4f}")
print(f"Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro")

FAIRNESS (robust): worst=0.3294, macro=0.1357
Compare: baseline=0.609 acc, 0.329 worst, 0.116 macro


In [9]:
SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")
cfg = {"method": f"Focal Loss gamma={GAMMA} + sqrt_rw", "gamma": GAMMA,
       "accuracy": float(accuracy_score(y_true, y_pred)), "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
       "tpr_gap_worst_robust": float(tw), "tpr_gap_macro_robust": float(tm)}
with open(SAVE_DIR / "training_config.json", "w") as f: json.dump(cfg, f, indent=2)
print(f"Saved: {SAVE_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.10s/it]

Saved: models/bert_focal_loss
